##Load and inspect Bronze lab test events

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "capstone_8hospital"
BRONZE = f"{CATALOG}.bronze_schema"
SILVER = f"{CATALOG}.silver_schema"
labs_bronze = spark.table(f"{BRONZE}.bronze_lab_test_events")
display(labs_bronze.limit(10))
print("Total lab event rows:", labs_bronze.count())

lab_event_id,encounter_id,patient_id,test_code,ordered_ts,collected_ts,result_ts,result_status,department_id,priority,_rescued_data,source_file,ingestion_timestamp
LAB_EVE080000001,ENCOUNT080000018,PATIEN0800135,GLUCOSE,2026-07-01T04:35:00Z,2026-07-01T04:40:00Z,2026-07-01T04:45:00Z,RESULTED,DEPART0800185,PRIORI_2,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/lab_test_events.csv,2026-11-04T06:14:39.801Z
LAB_EVE080000002,ENCOUNT080000035,PATIEN0800060,KFT,2026-07-01T04:42:00Z,2026-07-01T04:47:00Z,2026-07-01T04:52:00Z,COLLECTED,DEPART0800038,PRIORI_3,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/lab_test_events.csv,2026-11-04T06:14:39.801Z
LAB_EVE080000003,ENCOUNT080000052,PATIEN0800119,GLUCOSE,2026-07-01T04:49:00Z,2026-07-01T04:54:00Z,2026-07-01T04:59:00Z,ORDERED,DEPART0800081,PRIORI_4,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/lab_test_events.csv,2026-11-04T06:14:39.801Z
LAB_EVE080000004,ENCOUNT080000069,PATIEN0800049,KFT,2026-07-01T04:56:00Z,2026-07-01T05:01:00Z,2026-07-01T05:06:00Z,ORDERED,DEPART0800116,PRIORI_5,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/lab_test_events.csv,2026-11-04T06:14:39.801Z
LAB_EVE080000005,ENCOUNT080000086,PATIEN0800169,KFT,2026-07-01T05:03:00Z,2026-07-01T05:08:00Z,2026-07-01T05:13:00Z,COLLECTED,DEPART0800038,PRIORI_6,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/lab_test_events.csv,2026-11-04T06:14:39.801Z
LAB_EVE080000006,ENCOUNT080000103,PATIEN0800180,CBC,2026-07-01T05:10:00Z,2026-07-01T05:15:00Z,2026-07-01T05:20:00Z,RESULTED,DEPART0800120,PRIORI_7,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/lab_test_events.csv,2026-11-04T06:14:39.801Z
LAB_EVE080000007,ENCOUNT080000120,PATIEN0800116,CBC,2026-07-01T05:17:00Z,2026-07-01T05:22:00Z,2026-07-01T05:27:00Z,COLLECTED,DEPART0800147,PRIORI_8,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/lab_test_events.csv,2026-11-04T06:14:39.801Z
LAB_EVE080000008,ENCOUNT080000137,PATIEN0800051,KFT,2026-07-01T05:24:00Z,2026-07-01T05:29:00Z,2026-07-01T05:34:00Z,RESULTED,DEPART0800081,PRIORI_9,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/lab_test_events.csv,2026-11-04T06:14:39.801Z
LAB_EVE080000009,ENCOUNT080000154,PATIEN0800165,LFT,2026-07-01T05:31:00Z,2026-07-01T05:36:00Z,2026-07-01T05:41:00Z,RESULTED,DEPART0800162,PRIORI_10,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/lab_test_events.csv,2026-11-04T06:14:39.801Z
LAB_EVE080000010,ENCOUNT080000171,PATIEN0800037,GLUCOSE,2026-07-01T05:38:00Z,2026-07-01T05:43:00Z,2026-07-01T05:48:00Z,COLLECTED,DEPART0800106,PRIORI_11,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/lab_test_events.csv,2026-11-04T06:14:39.801Z


Total lab event rows: 758


##Clean lab test event columns

In [0]:
labs_checked = (
    labs_bronze
    .withColumn("lab_event_id", F.trim("lab_event_id"))
    .withColumn("encounter_id", F.trim("encounter_id"))
    .withColumn("patient_id", F.trim("patient_id"))
    .withColumn("test_code", F.upper(F.trim("test_code")))
    .withColumn("ordered_ts", F.to_timestamp("ordered_ts"))
    .withColumn("collected_ts", F.to_timestamp("collected_ts"))
    .withColumn("result_ts", F.to_timestamp("result_ts"))
    .withColumn("result_status", F.upper(F.trim("result_status")))
    .withColumn("department_id", F.trim("department_id"))
    .withColumn("priority", F.upper(F.trim("priority")))
)

##Check mandatory fields and timestamps

In [0]:
labs_checked = labs_checked.withColumn(
    "dq_reason",
    F.concat_ws(", ",
        F.when(F.col("lab_event_id").isNull() | (F.col("lab_event_id") == ""), "MISSING_LAB_EVENT_ID"),
        F.when(F.col("encounter_id").isNull() | (F.col("encounter_id") == ""), "MISSING_ENCOUNTER_ID"),
        F.when(F.col("ordered_ts").isNull(), "INVALID_ORDERED_TIMESTAMP"),
        F.when(F.col("collected_ts").isNotNull() & F.col("ordered_ts").isNotNull() & (F.col("collected_ts") < F.col("ordered_ts")), "COLLECTION_BEFORE_ORDER")
    )
)

labs_quarantine = labs_checked.filter(F.col("dq_reason") != "")
labs_valid = labs_checked.filter(F.col("dq_reason") == "")

##Validate encounter relationship

In [0]:
encounter_keys = spark.table(f"{SILVER}.silver_patient_encounters").select("encounter_id").distinct()

labs_valid = labs_valid.join(
    encounter_keys.withColumn("encounter_found", F.lit(1)), "encounter_id", "left"
).withColumn(
    "dq_reason",
    F.when(F.col("encounter_found").isNull(), "UNKNOWN_ENCOUNTER_ID").otherwise("")
)

labs_quarantine = labs_quarantine.unionByName(
    labs_valid.filter(F.col("dq_reason") != ""), allowMissingColumns=True
)
labs_valid = labs_valid.filter(F.col("dq_reason") == "")

##Deduplicate Lab Test Events

In [0]:
lab_window = Window.partitionBy("lab_event_id").orderBy(
    F.col("ingestion_timestamp").desc_nulls_last(),
    F.col("source_file").asc_nulls_last()
)

labs_ranked = labs_valid.withColumn("row_num", F.row_number().over(lab_window))
labs_duplicate_audit = labs_ranked.filter(F.col("row_num") > 1)
labs_silver = labs_ranked.filter(F.col("row_num") == 1).drop("row_num", "dq_reason", "encounter_found")

##Save Lab Silver, Quarantine and Duplicate Audit

In [0]:
(labs_silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.silver_lab_test_events"))

(labs_quarantine.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.silver_lab_quarantine"))

(labs_duplicate_audit.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.silver_lab_duplicate_audit"))

##Validate saved Lab tables

In [0]:
for table_name in ["silver_lab_test_events", "silver_lab_quarantine", "silver_lab_duplicate_audit"]:
    print(table_name, spark.table(f"{SILVER}.{table_name}").count())

display(spark.table(f"{SILVER}.silver_lab_test_events").limit(10))

silver_lab_test_events 758
silver_lab_quarantine 0
silver_lab_duplicate_audit 0


encounter_id,lab_event_id,patient_id,test_code,ordered_ts,collected_ts,result_ts,result_status,department_id,priority,_rescued_data,source_file,ingestion_timestamp
ENCOUNT080000018,LAB_EVE080000001,PATIEN0800135,GLUCOSE,2026-07-01T04:35:00.000Z,2026-07-01T04:40:00.000Z,2026-07-01T04:45:00.000Z,RESULTED,DEPART0800185,PRIORI_2,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/lab_test_events.csv,2026-11-04T06:14:39.801Z
ENCOUNT080000035,LAB_EVE080000002,PATIEN0800060,KFT,2026-07-01T04:42:00.000Z,2026-07-01T04:47:00.000Z,2026-07-01T04:52:00.000Z,COLLECTED,DEPART0800038,PRIORI_3,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/lab_test_events.csv,2026-11-04T06:14:39.801Z
ENCOUNT080000052,LAB_EVE080000003,PATIEN0800119,GLUCOSE,2026-07-01T04:49:00.000Z,2026-07-01T04:54:00.000Z,2026-07-01T04:59:00.000Z,ORDERED,DEPART0800081,PRIORI_4,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/lab_test_events.csv,2026-11-04T06:14:39.801Z
ENCOUNT080000069,LAB_EVE080000004,PATIEN0800049,KFT,2026-07-01T04:56:00.000Z,2026-07-01T05:01:00.000Z,2026-07-01T05:06:00.000Z,ORDERED,DEPART0800116,PRIORI_5,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/lab_test_events.csv,2026-11-04T06:14:39.801Z
ENCOUNT080000086,LAB_EVE080000005,PATIEN0800169,KFT,2026-07-01T05:03:00.000Z,2026-07-01T05:08:00.000Z,2026-07-01T05:13:00.000Z,COLLECTED,DEPART0800038,PRIORI_6,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/lab_test_events.csv,2026-11-04T06:14:39.801Z
ENCOUNT080000103,LAB_EVE080000006,PATIEN0800180,CBC,2026-07-01T05:10:00.000Z,2026-07-01T05:15:00.000Z,2026-07-01T05:20:00.000Z,RESULTED,DEPART0800120,PRIORI_7,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/lab_test_events.csv,2026-11-04T06:14:39.801Z
ENCOUNT080000120,LAB_EVE080000007,PATIEN0800116,CBC,2026-07-01T05:17:00.000Z,2026-07-01T05:22:00.000Z,2026-07-01T05:27:00.000Z,COLLECTED,DEPART0800147,PRIORI_8,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/lab_test_events.csv,2026-11-04T06:14:39.801Z
ENCOUNT080000137,LAB_EVE080000008,PATIEN0800051,KFT,2026-07-01T05:24:00.000Z,2026-07-01T05:29:00.000Z,2026-07-01T05:34:00.000Z,RESULTED,DEPART0800081,PRIORI_9,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/lab_test_events.csv,2026-11-04T06:14:39.801Z
ENCOUNT080000154,LAB_EVE080000009,PATIEN0800165,LFT,2026-07-01T05:31:00.000Z,2026-07-01T05:36:00.000Z,2026-07-01T05:41:00.000Z,RESULTED,DEPART0800162,PRIORI_10,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/lab_test_events.csv,2026-11-04T06:14:39.801Z
ENCOUNT080000171,LAB_EVE080000010,PATIEN0800037,GLUCOSE,2026-07-01T05:38:00.000Z,2026-07-01T05:43:00.000Z,2026-07-01T05:48:00.000Z,COLLECTED,DEPART0800106,PRIORI_11,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/lab_test_events.csv,2026-11-04T06:14:39.801Z
